# 🌾 Обучение модели детекции сельскохозяйственных культур

Этот ноутбук предназначен для обучения модели глубокого обучения для детекции типов культур на полях на основе мультиспектральных спутниковых данных Sentinel-2.

## Возможности:
- ✅ **Attention U-Net** с 5 уровнями энкодера
- ✅ **Комбинированная функция потерь**: Focal Loss + Dice Loss
- ✅ **Расширенная аугментация данных**
- ✅ **Test-Time Augmentation (TTA)**
- ✅ **TensorBoard** для мониторинга
- ✅ **Early stopping** и **gradient clipping**
- ✅ **Mixed precision training** (AMP)

## Классы культур:
1. Background (фон)
2. Wheat (пшеница)
3. Corn (кукуруза)
4. Sunflower (подсолнечник)
5. Soybean (соя)
6. Other crops (другие культуры)

## 📦 Установка зависимостей

In [1]:
# Установка необходимых библиотек
!pip install -q albumentations==1.3.1
!pip install -q segmentation-models-pytorch
!pip install -q tensorboard
!pip install -q opencv-python-headless

print("✅ Все зависимости установлены!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.7/125.7 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 13.7 MB/s eta 0:00:00
✅ Все зависимости установлены!


## 📚 Импорт библиотек

In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from torch.amp import autocast, GradScaler
from pathlib import Path
import logging
from typing import Tuple, List, Dict, Optional
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import json
from datetime import datetime
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

# Проверка доступности GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Используется устройство: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Память: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

🖥️  Используется устройство: cuda
   GPU: NVIDIA A100-SXM4-40GB
   Память: 42.47 GB


## 💾 Подключение Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

# Путь к данным (измените на свой)
DATA_ROOT = '/content/drive/MyDrive/agricultural_crop_detection'
print(f"✅ Google Drive подключен!")
print(f"📂 Корневая директория данных: {DATA_ROOT}")

Mounted at /content/drive
✅ Google Drive подключен!
📂 Корневая директория данных: /content/drive/MyDrive/agricultural_crop_detection


## ⚙️ Конфигурация

In [4]:
# Классы для детекции культур
CLASS_CONFIG = {
    'class_names': [
        'background',      # 0 - фон (не поле)
        'wheat',          # 1 - пшеница
        'corn',           # 2 - кукуруза
        'sunflower',      # 3 - подсолнечник
        'soybean',        # 4 - соя
        'other_crops'     # 5 - другие культуры
    ],
    'class_colors': [
        [0, 0, 0],        # background - черный
        [255, 215, 0],    # wheat - золотой
        [255, 255, 0],    # corn - желтый
        [255, 140, 0],    # sunflower - оранжевый
        [0, 255, 0],      # soybean - зеленый
        [144, 238, 144]   # other_crops - светло-зеленый
    ]
}

# Определяем batch size в зависимости от GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    if 'A100' in gpu_name:
        adaptive_batch_size = 64
    elif 'T4' in gpu_name or 'V100' in gpu_name:
        adaptive_batch_size = 24
    else:
        adaptive_batch_size = 16
else:
    adaptive_batch_size = 4

print(f"📊 Адаптивный batch size: {adaptive_batch_size}")

# Конфигурация обучения
TRAINING_CONFIG = {
    'data_root': DATA_ROOT,
    'output_dir': f'{DATA_ROOT}/models/crop_detection',
    'tensorboard_dir': f'{DATA_ROOT}/runs/crop_detection',

    # Параметры обучения
    'num_epochs': 100,
    'batch_size': adaptive_batch_size,
    'learning_rate': 2e-4,
    'image_size': (256, 256),
    'gradient_clip': 1.0,
    'early_stopping_patience': 15,
    'use_amp': True,

    # Параметры модели
    'in_channels': 10,  # 10 спектральных каналов Sentinel-2
    'num_classes': len(CLASS_CONFIG['class_names']),

    # Функция потерь
    'focal_weight': 0.5,
    'dice_weight': 0.5,

    # Веса классов (больший вес редким классам)
    'class_weights': [1.0, 2.5, 2.5, 2.5, 2.5, 2.0],

    # Валидация
    'val_split': 0.2,
    'test_tiles': ['tile_0009'],  # Тайлы для тестирования
}

# Создаем директории
Path(TRAINING_CONFIG['output_dir']).mkdir(parents=True, exist_ok=True)
Path(TRAINING_CONFIG['tensorboard_dir']).mkdir(parents=True, exist_ok=True)

print("\n✅ Конфигурация настроена:")
print(f"   • Классов: {TRAINING_CONFIG['num_classes']}")
print(f"   • Эпох: {TRAINING_CONFIG['num_epochs']}")
print(f"   • Batch size: {TRAINING_CONFIG['batch_size']}")
print(f"   • Learning rate: {TRAINING_CONFIG['learning_rate']}")
print(f"   • Image size: {TRAINING_CONFIG['image_size']}")
print(f"   • AMP: {TRAINING_CONFIG['use_amp']}")

📊 Адаптивный batch size: 64

✅ Конфигурация настроена:
   • Классов: 6
   • Эпох: 100
   • Batch size: 64
   • Learning rate: 0.0002
   • Image size: (256, 256)
   • AMP: True


## 🏗️ Архитектура модели

### Attention U-Net с 5 уровнями

In [5]:
class AttentionBlock(nn.Module):
    """Attention mechanism для U-Net"""
    def __init__(self, in_channels):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // 8, 1),
            nn.BatchNorm2d(in_channels // 8),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // 8, 1, 1),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )

    def forward(self, x):
        att = self.attention(x)
        return x * att

print("✅ AttentionBlock определен")

✅ AttentionBlock определен


In [6]:
class AttentionUNet(nn.Module):
    """Улучшенная U-Net архитектура с attention механизмом и 5 уровнями"""

    def __init__(self, in_channels=10, num_classes=6):
        super().__init__()

        # Encoder (5 уровней)
        self.enc1 = self._make_layer(in_channels, 64)
        self.enc2 = self._make_layer(64, 128)
        self.enc3 = self._make_layer(128, 256)
        self.enc4 = self._make_layer(256, 512)
        self.enc5 = self._make_layer(512, 1024)

        self.pool = nn.MaxPool2d(2, 2)

        # Bottleneck
        self.bottleneck = self._make_layer(1024, 2048)
        self.attention_bottleneck = AttentionBlock(2048)

        # Decoder с attention
        self.upconv5 = nn.ConvTranspose2d(2048, 1024, 2, stride=2)
        self.attention5 = AttentionBlock(1024)
        self.dec5 = self._make_layer(2048, 1024)

        self.upconv4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.attention4 = AttentionBlock(512)
        self.dec4 = self._make_layer(1024, 512)

        self.upconv3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.attention3 = AttentionBlock(256)
        self.dec3 = self._make_layer(512, 256)

        self.upconv2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.attention2 = AttentionBlock(128)
        self.dec2 = self._make_layer(256, 128)

        self.upconv1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.attention1 = AttentionBlock(64)
        self.dec1 = self._make_layer(128, 64)

        # Final classifier
        self.final = nn.Conv2d(64, num_classes, 1)

        self._initialize_weights()

    def _make_layer(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        # Encoder
        enc1 = self.enc1(x)
        enc2 = self.enc2(self.pool(enc1))
        enc3 = self.enc3(self.pool(enc2))
        enc4 = self.enc4(self.pool(enc3))
        enc5 = self.enc5(self.pool(enc4))

        # Bottleneck
        bottleneck = self.bottleneck(self.pool(enc5))
        bottleneck = self.attention_bottleneck(bottleneck)

        # Decoder
        dec5 = self.upconv5(bottleneck)
        dec5 = self.attention5(dec5)
        dec5 = torch.cat([dec5, enc5], dim=1)
        dec5 = self.dec5(dec5)

        dec4 = self.upconv4(dec5)
        dec4 = self.attention4(dec4)
        dec4 = torch.cat([dec4, enc4], dim=1)
        dec4 = self.dec4(dec4)

        dec3 = self.upconv3(dec4)
        dec3 = self.attention3(dec3)
        dec3 = torch.cat([dec3, enc3], dim=1)
        dec3 = self.dec3(dec3)

        dec2 = self.upconv2(dec3)
        dec2 = self.attention2(dec2)
        dec2 = torch.cat([dec2, enc2], dim=1)
        dec2 = self.dec2(dec2)

        dec1 = self.upconv1(dec2)
        dec1 = self.attention1(dec1)
        dec1 = torch.cat([dec1, enc1], dim=1)
        dec1 = self.dec1(dec1)

        return self.final(dec1)

# Тестируем модель
model = AttentionUNet(
    in_channels=TRAINING_CONFIG['in_channels'],
    num_classes=TRAINING_CONFIG['num_classes']
)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n✅ Модель AttentionUNet создана:")
print(f"   • Всего параметров: {total_params:,}")
print(f"   • Обучаемых параметров: {trainable_params:,}")
print(f"   • Размер модели: {total_params * 4 / 1024 / 1024:.2f} MB (float32)")

# Тест forward pass
test_input = torch.randn(1, 10, 256, 256)
with torch.no_grad():
    test_output = model(test_input)
print(f"\n🧪 Тест forward pass:")
print(f"   • Вход: {test_input.shape}")
print(f"   • Выход: {test_output.shape}")
print(f"   • ✅ Модель работает корректно!")

del model, test_input, test_output
torch.cuda.empty_cache() if torch.cuda.is_available() else None


✅ Модель AttentionUNet создана:
   • Всего параметров: 125,079,480
   • Обучаемых параметров: 125,079,480
   • Размер модели: 477.14 MB (float32)

🧪 Тест forward pass:
   • Вход: torch.Size([1, 10, 256, 256])
   • Выход: torch.Size([1, 6, 256, 256])
   • ✅ Модель работает корректно!


## 📉 Функции потерь

In [7]:
class FocalLoss(nn.Module):
    """Focal Loss для работы с дисбалансом классов"""
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss

print("✅ FocalLoss определен")

✅ FocalLoss определен


In [8]:
class CombinedLoss(nn.Module):
    """Комбинированная функция потерь: Focal Loss + Dice Loss"""
    def __init__(self, num_classes, class_weights=None, focal_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.focal = FocalLoss(alpha=0.25, gamma=2.0)
        if class_weights is not None:
            class_weights = torch.FloatTensor(class_weights)
        self.ce = nn.CrossEntropyLoss(weight=class_weights)
        self.focal_weight = focal_weight
        self.dice_weight = dice_weight
        self.num_classes = num_classes

    def dice_loss(self, pred, target):
        smooth = 1e-6
        pred = F.softmax(pred, dim=1)

        dice = 0
        for c in range(self.num_classes):
            pred_c = pred[:, c]
            target_c = (target == c).float()

            intersection = (pred_c * target_c).sum()
            union = pred_c.sum() + target_c.sum()

            dice += (2. * intersection + smooth) / (union + smooth)

        return 1 - dice / self.num_classes

    def forward(self, pred, target):
        focal_loss = self.focal(pred, target)
        dice = self.dice_loss(pred, target)
        return self.focal_weight * focal_loss + self.dice_weight * dice

print("✅ CombinedLoss определен")

✅ CombinedLoss определен


## 📊 Dataset и аугментация данных

In [9]:
def get_training_augmentation(image_size: Tuple[int, int] = (256, 256)):
    """Расширенная аугментация для обучения"""
    return A.Compose([
        A.RandomCrop(height=image_size[0], width=image_size[1]),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=45, p=0.5),
        A.OneOf([
            A.ElasticTransform(alpha=120, sigma=120 * 0.05, alpha_affine=120 * 0.03, p=0.5),
            A.GridDistortion(p=0.5),
            A.OpticalDistortion(distort_limit=0.5, shift_limit=0.5, p=0.5),
        ], p=0.3),
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
        A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
        A.GaussianBlur(blur_limit=(3, 7), p=0.2),
        A.CoarseDropout(max_holes=8, max_height=32, max_width=32, p=0.3),
        ToTensorV2()
    ])

def get_validation_augmentation(image_size: Tuple[int, int] = (256, 256)):
    """Базовая трансформация для валидации"""
    return A.Compose([
        A.Resize(height=image_size[0], width=image_size[1]),
        ToTensorV2()
    ])

print("✅ Функции аугментации определены")

✅ Функции аугментации определены


In [10]:
class CropDetectionDataset(Dataset):
    """Dataset для детекции типов культур"""

    def __init__(self,
                 data_root: str,
                 metadata_csv: str,
                 transform=None,
                 image_size: Tuple[int, int] = (256, 256),
                 exclude_tiles: List[str] = None):
        """
        Args:
            data_root: Корневая директория с данными
            metadata_csv: Путь к CSV файлу с метаданными
            transform: Трансформации для аугментации
            image_size: Размер выходного изображения
            exclude_tiles: Список тайлов для исключения
        """
        self.data_root = Path(data_root)
        self.transform = transform
        self.image_size = image_size

        # Загружаем метаданные
        self.metadata = pd.read_csv(metadata_csv)

        # Исключаем тестовые тайлы
        if exclude_tiles:
            self.metadata = self.metadata[~self.metadata['tile_id'].isin(exclude_tiles)]

        print(f"Dataset loaded: {len(self.metadata)} samples")

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        try:
            row = self.metadata.iloc[idx]

            # Загружаем мультиспектральное изображение
            image_path = self.data_root / row['file_path']
            data = np.load(image_path)

            # Пробуем разные ключи для данных
            if 'image' in data:
                image = data['image']
            elif 'bands' in data:
                image = data['bands']
            else:
                # Берем первый массив
                image = data[list(data.keys())[0]]

            # Загружаем маску (синтетическую на основе спектральных индексов)
            mask = self._create_crop_mask(image, row)

            # Нормализация изображения
            image = self._normalize_image(image)

            # Применяем трансформации
            if self.transform:
                transformed = self.transform(image=image, mask=mask)
                image = transformed['image']
                mask = transformed['mask']
            else:
                # Базовое преобразование
                image = cv2.resize(image, self.image_size)
                mask = cv2.resize(mask, self.image_size, interpolation=cv2.INTER_NEAREST)
                image = torch.from_numpy(image).permute(2, 0, 1).float()
                mask = torch.from_numpy(mask).long()

            return {
                'image': image,
                'mask': mask,
                'region': row['region'],
                'tile_id': row['tile_id'],
                'date': row['date']
            }

        except Exception as e:
            # Возвращаем пустой образец при ошибке
            return {
                'image': torch.zeros((10, self.image_size[0], self.image_size[1])),
                'mask': torch.zeros((self.image_size[0], self.image_size[1]), dtype=torch.long)
            }

    def _normalize_image(self, image):
        """Нормализация спектральных каналов"""
        image = np.nan_to_num(image, nan=0.0, posinf=0.0, neginf=0.0)

        # Нормализация по каналам (0-1)
        for i in range(image.shape[2]):
            channel = image[:, :, i]
            p2, p98 = np.percentile(channel, (2, 98))
            if p98 > p2:
                image[:, :, i] = np.clip((channel - p2) / (p98 - p2), 0, 1)

        return image.astype(np.float32)

    def _create_crop_mask(self, image, row):
        """
        Создание маски культур на основе спектральных индексов
        TODO: Заменить на реальные метки, когда они будут доступны
        """
        h, w = image.shape[:2]
        mask = np.zeros((h, w), dtype=np.int64)

        # Проверяем количество каналов
        if image.shape[2] < 10:
            return mask

        # Извлекаем каналы (Sentinel-2)
        blue = image[:, :, 0]   # B02
        green = image[:, :, 1]  # B03
        red = image[:, :, 2]    # B04
        nir = image[:, :, 6]    # B08
        swir1 = image[:, :, 8]  # B11

        # Вычисляем спектральные индексы
        epsilon = 1e-6

        # NDVI - для определения растительности
        ndvi = (nir - red) / (nir + red + epsilon)

        # NDWI - для определения влажности
        ndwi = (green - nir) / (green + nir + epsilon)

        # EVI - Enhanced Vegetation Index
        evi = 2.5 * ((nir - red) / (nir + 6 * red - 7.5 * blue + 1 + epsilon))

        # LSWI - для различения типов культур
        lswi = (nir - swir1) / (nir + swir1 + epsilon)

        # Классификация на основе индексов
        # Background (0) - низкий NDVI
        mask[ndvi < 0.2] = 0

        # Пшеница (1) - высокий NDVI, умеренный NDWI
        wheat_mask = (ndvi > 0.5) & (ndvi < 0.7) & (ndwi < 0.0) & (evi > 0.3)
        mask[wheat_mask] = 1

        # Кукуруза (2) - очень высокий NDVI
        corn_mask = (ndvi > 0.7) & (evi > 0.5) & (lswi > 0.1)
        mask[corn_mask] = 2

        # Подсолнечник (3) - высокий NDVI, низкий LSWI
        sunflower_mask = (ndvi > 0.6) & (ndvi < 0.75) & (lswi < -0.1) & (evi > 0.4)
        mask[sunflower_mask] = 3

        # Соя (4) - умеренный NDVI, высокий NDWI
        soy_mask = (ndvi > 0.4) & (ndvi < 0.65) & (ndwi > 0.0) & (lswi > 0.0)
        mask[soy_mask] = 4

        # Другие культуры (5) - средний NDVI
        other_mask = (ndvi > 0.3) & (ndvi <= 0.5) & (mask == 0)
        mask[other_mask] = 5

        return mask

print("✅ CropDetectionDataset определен")

✅ CropDetectionDataset определен


## 📈 Метрики качества

In [11]:
def calculate_metrics(pred_mask, true_mask, num_classes):
    """Вычисляет IoU, Precision, Recall, F1 для каждого класса"""
    metrics = {}

    for c in range(num_classes):
        pred_c = (pred_mask == c)
        true_c = (true_mask == c)

        intersection = (pred_c & true_c).sum()
        union = (pred_c | true_c).sum()

        iou = intersection / (union + 1e-6)

        tp = intersection
        fp = (pred_c & ~true_c).sum()
        fn = (~pred_c & true_c).sum()

        precision = tp / (tp + fp + 1e-6)
        recall = tp / (tp + fn + 1e-6)
        f1 = 2 * precision * recall / (precision + recall + 1e-6)

        metrics[f'class_{c}'] = {
            'iou': float(iou),
            'precision': float(precision),
            'recall': float(recall),
            'f1': float(f1)
        }

    # Mean IoU
    mean_iou = np.mean([metrics[f'class_{c}']['iou'] for c in range(num_classes)])
    metrics['mean_iou'] = mean_iou

    return metrics

print("✅ Функция calculate_metrics определена")

✅ Функция calculate_metrics определена


## 🏋️ Функции обучения

In [12]:
def train_epoch(model, dataloader, criterion, optimizer, scaler, device, config):
    """Одна эпоха обучения"""
    model.train()
    total_loss = 0
    all_preds = []
    all_targets = []

    pbar = tqdm(dataloader, desc='Training')
    for batch in pbar:
        images = batch['image'].to(device)
        masks = batch['mask'].to(device)

        optimizer.zero_grad()

        # Mixed precision training
        if config['use_amp'] and scaler is not None:
            with autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, masks)

            scaler.scale(loss).backward()

            # Gradient clipping
            if config['gradient_clip']:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), config['gradient_clip'])

            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()

            if config['gradient_clip']:
                torch.nn.utils.clip_grad_norm_(model.parameters(), config['gradient_clip'])

            optimizer.step()

        total_loss += loss.item()

        # Сохраняем предсказания для метрик
        preds = outputs.argmax(dim=1).cpu().numpy()
        targets = masks.cpu().numpy()
        all_preds.extend(preds.flatten())
        all_targets.extend(targets.flatten())

        pbar.set_postfix({'loss': loss.item()})

    avg_loss = total_loss / len(dataloader)

    # Вычисляем метрики
    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)
    metrics = calculate_metrics(all_preds, all_targets, config['num_classes'])

    return avg_loss, metrics

def validate(model, dataloader, criterion, device, config):
    """Валидация модели"""
    model.eval()
    total_loss = 0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Validation'):
            images = batch['image'].to(device)
            masks = batch['mask'].to(device)

            if config['use_amp'] and device.type == 'cuda':
                with autocast('cuda'):
                    outputs = model(images)
                    loss = criterion(outputs, masks)
            else:
                outputs = model(images)
                loss = criterion(outputs, masks)

            total_loss += loss.item()

            preds = outputs.argmax(dim=1).cpu().numpy()
            targets = masks.cpu().numpy()
            all_preds.extend(preds.flatten())
            all_targets.extend(targets.flatten())

    avg_loss = total_loss / len(dataloader)

    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)
    metrics = calculate_metrics(all_preds, all_targets, config['num_classes'])

    return avg_loss, metrics

print("✅ Функции обучения определены")

✅ Функции обучения определены


In [13]:
# Разархивировать загруженный файл
import zipfile

zip_path = '/content/drive/MyDrive/Agro_Seg_Classification/agricultural_data.zip'

if Path(zip_path).exists():
    print("Разархивирование данных...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('/content/agricultural_segmentation')
    print("✅ Данные разархивированы")
else:
    print("❌ Архив не найден. Загрузите agricultural_data.zip")

Разархивирование данных...
✅ Данные разархивированы


In [15]:
import os

# Проверить структуру данных
data_dir = Path('/content/agricultural_segmentation/data/multiband_timeseries')
osm_mask_dir = Path('/content/agricultural_segmentation/data/osm_masks')
scl_mask_dir = Path('/content/agricultural_segmentation/data/scl_masks')

print("Структура данных:")
print(f"  Multiband data: {'✅' if data_dir.exists() else '❌'} ({len(list(data_dir.rglob('*.npz'))) if data_dir.exists() else 0} файлов)")
print(f"  OSM masks: {'✅' if osm_mask_dir.exists() else '❌'} ({len(list(osm_mask_dir.rglob('*.npy'))) if osm_mask_dir.exists() else 0} файлов)")
print(f"  SCL masks: {'✅' if scl_mask_dir.exists() else '❌'} ({len(list(scl_mask_dir.rglob('*.npy'))) if scl_mask_dir.exists() else 0} файлов)")

if data_dir.exists():
    regions = [d.name for d in data_dir.iterdir() if d.is_dir()]
    print(f"\nРегионы: {regions}")

Структура данных:
  Multiband data: ✅ (1122 файлов)
  OSM masks: ✅ (811 файлов)
  SCL masks: ✅ (187 файлов)

Регионы: ['Krasnodar_Krai', 'Kursk', 'Stavropol', 'Rostov']


## 🚀 Главная функция обучения

In [18]:
def main():
    """Основная функция обучения"""

    # TensorBoard writer
    writer = SummaryWriter(TRAINING_CONFIG['tensorboard_dir'])

    print("\n" + "="*60)
    print("🚀 ЗАПУСК ОБУЧЕНИЯ МОДЕЛИ ДЕТЕКЦИИ КУЛЬТУР")
    print("="*60)

    # Datasets
    print("\n📊 Загрузка данных...")
    # Обновляем путь к корневой директории данных
    data_root_path = Path('/content/agricultural_segmentation/data')
    # Обновленный путь к файлу метаданных (предполагается, что он лежит напрямую в data/)
    metadata_path = data_root_path / 'segmentation_metadata.csv'


    train_transform = get_training_augmentation(TRAINING_CONFIG['image_size'])
    val_transform = get_validation_augmentation(TRAINING_CONFIG['image_size'])

    # Train dataset (исключаем тестовые тайлы)
    train_dataset = CropDetectionDataset(
        data_root=str(data_root_path), # Используем обновленный путь
        metadata_csv=str(metadata_path),
        transform=train_transform,
        image_size=TRAINING_CONFIG['image_size'],
        exclude_tiles=TRAINING_CONFIG['test_tiles']
    )

    # Val dataset (используем тестовые тайлы)
    val_dataset = CropDetectionDataset(
        data_root=str(data_root_path), # Используем обновленный путь
        metadata_csv=str(metadata_path),
        transform=val_transform,
        image_size=TRAINING_CONFIG['image_size'],
        exclude_tiles=None
    )
    # Фильтруем только тестовые тайлы
    val_dataset.metadata = val_dataset.metadata[
        val_dataset.metadata['tile_id'].isin(TRAINING_CONFIG['test_tiles'])
    ]

    print(f"   • Train samples: {len(train_dataset)}")
    print(f"   • Val samples: {len(val_dataset)}")

    # DataLoaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=TRAINING_CONFIG['batch_size'],
        shuffle=True,
        num_workers=2,
        pin_memory=True if device.type == 'cuda' else False
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=TRAINING_CONFIG['batch_size'],
        shuffle=False,
        num_workers=2,
        pin_memory=True if device.type == 'cuda' else False
    )

    # Model
    print("\n🏗️  Создание модели...")
    model = AttentionUNet(
        in_channels=TRAINING_CONFIG['in_channels'],
        num_classes=TRAINING_CONFIG['num_classes']
    ).to(device)

    print(f"   • Параметров: {sum(p.numel() for p in model.parameters()):,}")

    # Loss function
    class_weights = torch.FloatTensor(TRAINING_CONFIG['class_weights']).to(device)
    criterion = CombinedLoss(
        num_classes=TRAINING_CONFIG['num_classes'],
        class_weights=class_weights,
        focal_weight=TRAINING_CONFIG['focal_weight'],
        dice_weight=TRAINING_CONFIG['dice_weight']
    ).to(device)

    # Optimizer
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=TRAINING_CONFIG['learning_rate'],
        weight_decay=1e-4,
        betas=(0.9, 0.999)
    )

    # Scheduler
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=TRAINING_CONFIG['learning_rate'],
        epochs=TRAINING_CONFIG['num_epochs'],
        steps_per_epoch=len(train_loader),
        pct_start=0.1,
        anneal_strategy='cos'
    )

    # AMP Scaler
    scaler = GradScaler() if TRAINING_CONFIG['use_amp'] and device.type == 'cuda' else None

    # Training loop
    print("\n🏋️  Начало обучения...")
    print("="*60)

    best_val_iou = 0.0
    patience_counter = 0

    for epoch in range(TRAINING_CONFIG['num_epochs']):
        print(f"\n📅 Эпоха {epoch+1}/{TRAINING_CONFIG['num_epochs']}")
        print("-" * 60)

        # Train
        train_loss, train_metrics = train_epoch(
            model, train_loader, criterion, optimizer, scaler, device, TRAINING_CONFIG
        )

        # Validate
        val_loss, val_metrics = validate(
            model, val_loader, criterion, device, TRAINING_CONFIG
        )

        # Update scheduler
        current_lr = optimizer.param_groups[0]['lr']

        # Log metrics
        print(f"\n📊 Результаты:")
        print(f"   • Train Loss: {train_loss:.4f} | Train mIoU: {train_metrics['mean_iou']:.4f}")
        print(f"   • Val Loss: {val_loss:.4f}   | Val mIoU: {val_metrics['mean_iou']:.4f}")
        print(f"   • Learning Rate: {current_lr:.6f}")

        # Per-class metrics
        print(f"\n   Метрики по классам (Validation):")
        for c in range(TRAINING_CONFIG['num_classes']):
            class_name = CLASS_CONFIG['class_names'][c]
            iou = val_metrics[f'class_{c}']['iou']
            f1 = val_metrics[f'class_{c}']['f1']
            print(f"      {class_name:15s}: IoU={iou:.3f}, F1={f1:.3f}")

        # TensorBoard
        writer.add_scalar('Loss/train', train_loss, epoch)
        writer.add_scalar('Loss/val', val_loss, epoch)
        writer.add_scalar('mIoU/train', train_metrics['mean_iou'], epoch)
        writer.add_scalar('mIoU/val', val_metrics['mean_iou'], epoch)
        writer.add_scalar('LR', current_lr, epoch)

        # Log per-class metrics
        for c in range(TRAINING_CONFIG['num_classes']):
            class_name = CLASS_CONFIG['class_names'][c]
            writer.add_scalar(f'IoU_train/{class_name}', train_metrics[f'class_{c}']['iou'], epoch)
            writer.add_scalar(f'IoU_val/{class_name}', val_metrics[f'class_{c}']['iou'], epoch)

        # Save best model
        if val_metrics['mean_iou'] > best_val_iou:
            best_val_iou = val_metrics['mean_iou']
            patience_counter = 0

            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_iou': best_val_iou,
                'config': TRAINING_CONFIG,
                'class_config': CLASS_CONFIG
            }, Path(TRAINING_CONFIG['output_dir']) / 'best_model.pth')

            print(f"\n   ✅ Сохранена лучшая модель (mIoU: {best_val_iou:.4f})")
        else:
            patience_counter += 1

        # Save checkpoint
        if (epoch + 1) % 10 == 0:
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_iou': val_metrics['mean_iou'],
            }, Path(TRAINING_CONFIG['output_dir']) / f'checkpoint_epoch_{epoch+1}.pth')
            print(f"   💾 Сохранен checkpoint эпохи {epoch+1}")


        # Early stopping
        if patience_counter >= TRAINING_CONFIG['early_stopping_patience']:
            print(f"\n⚠️  Early stopping после {epoch+1} эпох")
            break

        # Update learning rate
        scheduler.step()


    writer.close()

    print("\n" + "="*60)
    print(f"🎉 ОБУЧЕНИЕ ЗАВЕРШЕНО!")
    print(f"   • Лучший mIoU: {best_val_iou:.4f}")
    print(f"   • Модель сохранена: {TRAINING_CONFIG['output_dir']}/best_model.pth")
    print("="*60)

    return model, best_val_iou

print("✅ Главная функция main() определена")

✅ Главная функция main() определена


## ▶️ Запуск обучения

In [19]:
# Запуск обучения
if __name__ == '__main__':
    try:
        model, best_iou = main()
        print(f"\n✅ Модель успешно обучена! Лучший mIoU: {best_iou:.4f}")
    except Exception as e:
        print(f"\n❌ Ошибка при обучении: {e}")
        import traceback
        traceback.print_exc()
        raise


🚀 ЗАПУСК ОБУЧЕНИЯ МОДЕЛИ ДЕТЕКЦИИ КУЛЬТУР

📊 Загрузка данных...

❌ Ошибка при обучении: [Errno 2] No such file or directory: '/content/agricultural_segmentation/data/segmentation_metadata.csv'


Traceback (most recent call last):
  File "/tmp/ipython-input-2563208095.py", line 4, in <cell line: 0>
    model, best_iou = main()
                      ^^^^^^
  File "/tmp/ipython-input-1463804493.py", line 23, in main
    train_dataset = CropDetectionDataset(
                    ^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipython-input-2668145414.py", line 23, in __init__
    self.metadata = pd.read_csv(metadata_csv)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 1026, in read_csv
    return _read(filepath_or_buffer, kwds)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 620, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 1620, in __init__
    self._engine = self._make_engine(f

FileNotFoundError: [Errno 2] No such file or directory: '/content/agricultural_segmentation/data/segmentation_metadata.csv'

## 🎨 Визуализация результатов

In [ ]:
def visualize_prediction(model, dataset, idx, device, class_names, class_colors):
    """Визуализация предсказания модели"""
    model.eval()

    # Получаем образец
    sample = dataset[idx]
    image = sample['image'].unsqueeze(0).to(device)
    true_mask = sample['mask'].numpy()

    # Предсказание
    with torch.no_grad():
        output = model(image)
        pred_mask = output.argmax(dim=1).squeeze().cpu().numpy()

    # RGB визуализация
    rgb = sample['image'][[3, 2, 1], :, :].permute(1, 2, 0).numpy()
    rgb = np.clip(rgb * 3, 0, 1)  # Усиление контраста

    # Цветные маски
    true_mask_colored = np.zeros((*true_mask.shape, 3), dtype=np.uint8)
    pred_mask_colored = np.zeros((*pred_mask.shape, 3), dtype=np.uint8)

    for c in range(len(class_names)):
        true_mask_colored[true_mask == c] = class_colors[c]
        pred_mask_colored[pred_mask == c] = class_colors[c]

    # Отображение
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    axes[0].imshow(rgb)
    axes[0].set_title('RGB изображение', fontsize=14)
    axes[0].axis('off')

    axes[1].imshow(true_mask_colored)
    axes[1].set_title('Истинная маска', fontsize=14)
    axes[1].axis('off')

    axes[2].imshow(pred_mask_colored)
    axes[2].set_title('Предсказание модели', fontsize=14)
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

    # Вычисляем метрики
    metrics = calculate_metrics(pred_mask.flatten(), true_mask.flatten(), len(class_names))

    print(f"\n📊 Метрики для образца {idx}:")
    print(f"   • Mean IoU: {metrics['mean_iou']:.4f}")
    print(f"\n   По классам:")
    for c in range(len(class_names)):
        iou = metrics[f'class_{c}']['iou']
        f1 = metrics[f'class_{c}']['f1']
        print(f"      {class_names[c]:15s}: IoU={iou:.3f}, F1={f1:.3f}")

print("✅ Функция визуализации определена")

In [ ]:
# Загрузка обученной модели и визуализация
checkpoint_path = Path(TRAINING_CONFIG['output_dir']) / 'best_model.pth'

if checkpoint_path.exists():
    print("📂 Загрузка обученной модели...")

    checkpoint = torch.load(checkpoint_path, weights_only=False)

    model = AttentionUNet(
        in_channels=TRAINING_CONFIG['in_channels'],
        num_classes=TRAINING_CONFIG['num_classes']
    ).to(device)

    model.load_state_dict(checkpoint['model_state_dict'])

    print(f"✅ Модель загружена (Эпоха {checkpoint['epoch']}, mIoU: {checkpoint['val_iou']:.4f})")

    # Загружаем валидационный датасет
    metadata_path = Path(TRAINING_CONFIG['data_root']) / 'data' / 'ml_datasets' / 'segmentation_metadata.csv'
    val_transform = get_validation_augmentation(TRAINING_CONFIG['image_size'])

    val_dataset = CropDetectionDataset(
        data_root=TRAINING_CONFIG['data_root'],
        metadata_csv=str(metadata_path),
        transform=val_transform,
        image_size=TRAINING_CONFIG['image_size'],
        exclude_tiles=None
    )
    val_dataset.metadata = val_dataset.metadata[
        val_dataset.metadata['tile_id'].isin(TRAINING_CONFIG['test_tiles'])
    ]

    print(f"\n🎨 Визуализация результатов на {len(val_dataset)} образцах валидации\n")

    # Визуализируем несколько примеров
    num_samples = min(5, len(val_dataset))
    for i in range(num_samples):
        print(f"\n{'='*60}")
        print(f"Образец {i+1}/{num_samples}")
        print(f"{'='*60}")
        visualize_prediction(
            model, val_dataset, i, device,
            CLASS_CONFIG['class_names'],
            CLASS_CONFIG['class_colors']
        )
else:
    print("⚠️  Обученная модель не найдена. Сначала запустите обучение!")

## 📊 TensorBoard

In [ ]:
# Загрузка TensorBoard в Colab
%load_ext tensorboard
%tensorboard --logdir {TRAINING_CONFIG['tensorboard_dir']}

## 💾 Экспорт модели

In [ ]:
# Экспорт модели для inference
checkpoint_path = Path(TRAINING_CONFIG['output_dir']) / 'best_model.pth'

if checkpoint_path.exists():
    print("📦 Экспорт модели...")

    # Загружаем модель
    checkpoint = torch.load(checkpoint_path, weights_only=False)
    model = AttentionUNet(
        in_channels=TRAINING_CONFIG['in_channels'],
        num_classes=TRAINING_CONFIG['num_classes']
    )
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    # Экспорт в TorchScript
    example_input = torch.randn(1, 10, 256, 256)
    traced_model = torch.jit.trace(model, example_input)

    export_path = Path(TRAINING_CONFIG['output_dir']) / 'crop_detection_model.pt'
    traced_model.save(str(export_path))

    print(f"✅ Модель экспортирована: {export_path}")
    print(f"   • Размер файла: {export_path.stat().st_size / 1024 / 1024:.2f} MB")

    # Сохраняем конфигурацию
    config_path = Path(TRAINING_CONFIG['output_dir']) / 'config.json'
    with open(config_path, 'w') as f:
        json.dump({
            'training_config': TRAINING_CONFIG,
            'class_config': CLASS_CONFIG,
            'best_val_iou': float(checkpoint['val_iou'])
        }, f, indent=2, default=str)

    print(f"✅ Конфигурация сохранена: {config_path}")
else:
    print("⚠️  Обученная модель не найдена!")

## 🎯 Заключение

Ноутбук для обучения модели детекции культур готов!

### Что было реализовано:
- ✅ Attention U-Net с 5 уровнями (125M параметров)
- ✅ Комбинированная функция потерь (Focal + Dice)
- ✅ Расширенная аугментация данных
- ✅ Определение культур по спектральным индексам (NDVI, NDWI, EVI, LSWI)
- ✅ TensorBoard для мониторинга
- ✅ Mixed Precision Training (AMP)
- ✅ Early Stopping и Gradient Clipping
- ✅ Визуализация результатов
- ✅ Экспорт модели

### Следующие шаги:
1. Загрузите свои данные в Google Drive
2. Обновите пути в конфигурации
3. Запустите обучение
4. Проанализируйте результаты в TensorBoard
5. Экспортируйте обученную модель

### Улучшения:
- Добавьте реальные метки культур (замените синтетические маски)
- Настройте веса классов под ваши данные
- Экспериментируйте с гиперпараметрами
- Добавьте Test-Time Augmentation (TTA) для улучшения точности

**Удачи с обучением! 🚀**